# Build the guide-module object

**Run this notebook before `Figure3_B`.** It writes
`outputs/adata_SingleKO_GuideModules.h5ad`, which `Figure3_B` reads.

`Figure3_B` needs a per-cell indicator of which *guide module* a cell was
perturbed in (`M1` … `M6`, and `CONTROL` for controls). Those columns were
added by a separate pipeline, which is why that figure used to read an object
no other notebook touches.

This notebook rebuilds them from
`adata-hash-features_singlets_05242020.h5ad` — the screen object the other
figure notebooks read — reproducing the pipeline that originally produced them:

| Step | Source notebook |
|---|---|
| keep single-knockout cells | `07_01_splitSingleKOComb` |
| filter genes and cells, drop outlier control guides | `07_04_furtherFilterGenes` |
| drop bad knockout guides, aggregate guides to genes | `07_06_ReduceToGenes` |
| restrict to the module gene set, assign modules | `08-01-ReduceAnndataToSelectedKOsGenes` |

UMAP coordinates and leiden labels are carried through untouched, so the
embedding is the same one the original figure used.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = "hint"

In [2]:
E3LIGASE = Path("/home/eraslab1/Projects/E3Ligase/analysisSingle")

SCREEN_H5AD      = E3LIGASE / "outputs/anndata/OriginalFiles/adata-hash-features_singlets_05242020.h5ad"
GENE_MODULES_8   = E3LIGASE / "TextFiles/ME_GeneModules_leiden_8_Modules.csv"
GUIDE_MODULES    = E3LIGASE / "TextFiles/ME_GuideModules_leiden_6_Modules.csv"
BAD_KO_GUIDES    = E3LIGASE / "R/GuideSelect_BadKOGuides.csv"
OUTLIER_CONTROLS = E3LIGASE / "TextFiles/OutlierControlGuides.csv"

OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_H5AD = OUTPUT_DIR / "adata_SingleKO_GuideModules.h5ad"

CONTROL_PREFIXES    = ("NO_TARGET_", "ONE_NONGENE_SITE_")
MIN_CELLS_PER_GENE  = 20000   # 07_04: genes must be seen this widely
MIN_GENES_PER_CELL  = 800     # 07_04
MIN_CELLS_PER_GUIDE = 20      # 07_04 and 07_06

# genes the original pipeline appended to the module list
EXTRA_GENES = [
    "0610012G03Rik", "2010005H15Rik", "2010111I01Rik", "2310001H17Rik",
    "2810474O19Rik", "H2-Q7", "H2-Q6", "H2-DMa", "H2-T23", "H2-DMb1",
    "H2-Ab1", "H2-Aa", "H2-Eb1", "H2-M2", "H2-K1", "H2-D1",
]

## Keep single-knockout cells

`07_01_splitSingleKOComb`: a cell's knockout count is how many guides it
carries; cells carrying more than one are the multiple-knockout population and
are analysed separately.

In [3]:
adata = sc.read(SCREEN_H5AD)
print(f"screen: {adata.shape[0]} cells x {adata.shape[1]} genes")

guideNames = list(adata.uns["feature_barcode_names"])
adata.obs["KONo"] = (adata.obs[guideNames] > 0).sum(axis=1).to_numpy()
adata = adata[adata.obs.KONo <= 1].copy()
print(f"single-knockout cells: {adata.shape[0]}")

/home/eraslab1/miniconda3/lib/python3.8/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/eraslab1/miniconda3/lib/python3.8/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


screen: 519535 cells x 13811 genes
single-knockout cells: 341664


## Filter genes and cells, and drop outlier control guides

`07_04_furtherFilterGenes`. The gene filter runs before the cell filter, so
the detected-gene count is measured on the reduced gene set.

In [4]:
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
print(f"after gene/cell filtering: {adata.shape[0]} cells x {adata.shape[1]} genes")

guides = adata.obs[guideNames] > 0
outlierControls = set(pd.read_csv(OUTLIER_CONTROLS)["OutlierGuides"])
drop = (guides.sum(axis=0) < MIN_CELLS_PER_GUIDE) | guides.columns.isin(outlierControls)
guideNames = [g for g in guideNames if not drop[g]]
print(f"guides kept after dropping outlier controls: {len(guideNames)}")

adata = adata[(adata.obs[guideNames] > 0).sum(axis=1) > 0].copy()
print(f"cells still carrying a guide: {adata.shape[0]}")

filtered out 7126 genes that are detected in less than 20000 cells
filtered out 13007 cells that have less than 800 genes expressed
after gene/cell filtering: 328657 cells x 6685 genes
guides kept after dropping outlier controls: 3502
cells still carrying a guide: 325203


## Drop bad knockout guides

`07_06_ReduceToGenes`: guides that the guide-selection step found not to work,
plus any guide left in too few cells after the previous filtering.

In [5]:
guides = adata.obs[guideNames] > 0
badGuides = set(pd.read_csv(BAD_KO_GUIDES)["x"])
drop = (guides.sum(axis=0) < MIN_CELLS_PER_GUIDE) | guides.columns.isin(badGuides)
guideNames = [g for g in guideNames if not drop[g]]
print(f"guides kept: {len(guideNames)}")

adata = adata[(adata.obs[guideNames] > 0).sum(axis=1) > 0].copy()
print(f"cells still carrying a guide: {adata.shape[0]}")

guides kept: 2561
cells still carrying a guide: 246050


## Aggregate guides to their target gene

A cell is perturbed in a gene if it carries any surviving guide against it, and
counts as a control if its dominant guide is a control guide.

In [6]:
guides = adata.obs[guideNames] > 0
isControl = np.array([g.startswith(CONTROL_PREFIXES) for g in guideNames])

targetGene = pd.Series([g.rsplit("_", 1)[0] for g in guideNames], index=guideNames)
perGene = guides.loc[:, ~isControl].groupby(targetGene[~isControl], axis=1).any()
perGene["CONTROL"] = pd.Series(isControl[guides.values.argmax(axis=1)], index=guides.index)

print(f"target genes: {perGene.shape[1] - 1}")

target genes: 1031


## Restrict to the module gene set and assign each cell its module

In [7]:
moduleGenes = list(pd.read_csv(GENE_MODULES_8, index_col=0).iloc[:, 0])
keepGenes = [g for g in moduleGenes + EXTRA_GENES if g in adata.var_names]
adata = adata[:, keepGenes].copy()
print(f"reduced to {adata.shape[1]} genes")

guideModules = pd.read_csv(GUIDE_MODULES, index_col=0)
# modules are named by NewGuideGroup (M1..M6); GuideGroup holds the older K
# numbering, kept here only to read the table
moduleOf = dict(zip(guideModules.GuideName, guideModules.NewGuideGroup))
MODULES = sorted(set(moduleOf.values()))

inModule = [g for g in perGene.columns if g in moduleOf]
outsideModule = [g for g in perGene.columns if g not in moduleOf and g != "CONTROL"]

nTargets = perGene[inModule + ["CONTROL"]].sum(axis=1)
onlyModuleGuides = (nTargets > 0) & (perGene[outsideModule].sum(axis=1) == 0)
singleTarget = onlyModuleGuides & (nTargets == 1)
print(f"modules: {MODULES}")
print(f"cells carrying only module guides: {int(onlyModuleGuides.sum())}")
print(f"  of which a single target       : {int(singleTarget.sum())}")

reduced to 1041 genes
modules: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']
cells carrying only module guides: 122589
  of which a single target       : 122589


In [8]:
for module in MODULES:
    genesInModule = [g for g in inModule if moduleOf[g] == module]
    adata.obs[module] = (perGene[genesInModule].any(axis=1) & singleTarget).astype(int)
adata.obs["CONTROL"] = (perGene["CONTROL"] & singleTarget).astype(int)

adata = adata[singleTarget.values].copy()

moduleColumns = MODULES + ["CONTROL"]
print(adata.obs[moduleColumns].sum().to_string())
print(f"\nfinal object: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"UMAP carried through: {'X_umap' in adata.obsm}")

M1          4743
M2         22988
M3         17808
M4          1839
M5         17417
M6         16671
CONTROL    41123

final object: 122589 cells x 1041 genes
UMAP carried through: True


In [9]:
adata.write(OUTPUT_H5AD)
print(f"written to {OUTPUT_H5AD.resolve()}")

written to /home/eraslab1/Projects/PerturbDecode/notebooks/manuscript_figures/outputs/adata_SingleKO_GuideModules.h5ad
